# spVIPES: Multi-group Integration on DIALOGUE Example Data

This vignette showcases **spVIPES' multi-group (N ≥ 2) integration** by
analysing the same PBMC dataset used in the
[pertpy DIALOGUE tutorial](https://pertpy.readthedocs.io/en/stable/tutorials/notebooks/dialogue.html).

**DIALOGUE** (Jerber *et al.* 2021; Jerby-Arnon & Regev 2022) finds
*multicellular programs* — coordinated gene-expression variation across
cell types — that differ between **sample groups** (e.g. disease vs.
control). It groups cells by a **clinical / sample-level covariate**
such as `clinical.status`.

**spVIPES** solves a related problem with a different model:
a **Product-of-Experts VAE** that learns a **shared** latent (biology
common to every group, such as cell-type identity) and a **private**
latent per group (group-specific variation, such as disease programs).
Crucially, the label-based PoE variant supports **any number of groups
N ≥ 2** — not just pairwise comparisons.

In this vignette we

| Step | What we do |
|---|---|
| §1 | Load the DIALOGUE example via `pertpy` |
| §2 | Inspect cell types and the `clinical.status` grouping |
| §3 | Split the data by `clinical.status` into N groups |
| §4 | Prepare the multi-group AnnData with `prepare_adatas` |
| §5 | Train spVIPES with label-based PoE over all groups |
| §6 | Visualise the **shared** latent (cell-type identity across groups) |
| §7 | Visualise the **private** latent (group-specific variation) |
| §8 | Summary and connection to DIALOGUE |


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import torch

import pertpy as pt
import spVIPES

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
sc.settings.set_figure_params(dpi=100, frameon=False)

## 1. Load the DIALOGUE example dataset

`pt.dt.dialogue_example()` returns a single-cell PBMC dataset used in
the original DIALOGUE publication. It carries sample-level clinical
annotations — including the `clinical.status` grouping we will use to
define spVIPES groups — and cell-type labels in `cell.subtypes`.

In [ ]:
adata = pt.dt.dialogue_example()
print(f"Shape                : {adata.shape}")
print(f"obs columns          : {list(adata.obs.columns)}")
adata

## 2. Inspect cell types and the `clinical.status` groups

spVIPES' multi-group setting requires

1. a **group** covariate — here `clinical.status`, which defines the
   patient cohorts we want to integrate (one private latent per cohort);
2. a **label** covariate for the label-based Product-of-Experts —
   here `cell.subtypes`, the cell-type annotation shared across cohorts.


In [ ]:
GROUP_KEY = "clinical.status"
LABEL_KEY = "cell.subtypes"

print("Groups (clinical.status):")
print(adata.obs[GROUP_KEY].value_counts())
print()
print("Cell types (cell.subtypes):")
print(adata.obs[LABEL_KEY].value_counts())

Notice that `clinical.status` can hold **more than two** levels —
spVIPES will learn one private latent for each of them. This is the
core multi-group capability that generalises the two-group perturbation
setup shown in the other vignettes.

In [ ]:
# Light preprocessing — keep only annotated cells and drop any NaNs in the keys we need
adata = adata[adata.obs[GROUP_KEY].notna() & adata.obs[LABEL_KEY].notna()].copy()

# Make categoricals tidy
adata.obs[GROUP_KEY] = adata.obs[GROUP_KEY].astype("category")
adata.obs[LABEL_KEY] = adata.obs[LABEL_KEY].astype("category")

# Raw counts: spVIPES uses a negative-binomial likelihood, so we need counts.
# The DIALOGUE example ships log-normalised data; we recover integer counts
# from adata.layers["counts"] if present, otherwise from adata.raw.
if "counts" in adata.layers:
    adata.X = adata.layers["counts"].copy()
elif adata.raw is not None:
    adata = adata.raw.to_adata()

# HVG selection on a log-normalised copy so training stays lightweight.
adata_norm = adata.copy()
sc.pp.normalize_total(adata_norm, target_sum=1e4)
sc.pp.log1p(adata_norm)
sc.pp.highly_variable_genes(adata_norm, n_top_genes=2000, flavor="seurat", batch_key=GROUP_KEY)
adata = adata[:, adata_norm.var["highly_variable"].values].copy()
print(f"After HVG selection  : {adata.shape}")

## 3. Split the data by `clinical.status` into N groups

We create one AnnData per clinical status. `prepare_adatas` will then
stack them into the combined object that spVIPES expects, storing the
per-group index slices in `.uns`.

In [ ]:
group_levels = list(adata.obs[GROUP_KEY].cat.categories)
print(f"Groups to integrate  : {group_levels}")

group_adatas = {
    g: adata[adata.obs[GROUP_KEY] == g].copy()
    for g in group_levels
}
for g, a in group_adatas.items():
    print(f"  {g:>20s}  {a.shape}")

## 4. Prepare the multi-group AnnData

`spVIPES.data.prepare_adatas` concatenates the per-group objects and
stores the per-group indices, variable names and lengths in `.uns`
so spVIPES can dispatch encoders/decoders per group at training time.

In [ ]:
sdata = spVIPES.data.prepare_adatas(group_adatas)

print(f"Combined shape        : {sdata.shape}")
print(f"Groups column         : {sdata.obs['groups'].value_counts().to_dict()}")
print(f"Groups mapping (.uns) : {sdata.uns['groups_mapping']}")

In [ ]:
spVIPES.model.spVIPES.setup_anndata(
    sdata,
    groups_key="groups",
    label_key=LABEL_KEY,
)

group_indices_list = [
    np.where(sdata.obs["groups"] == g)[0]
    for g in sdata.obs["groups"].cat.categories
]
for g, idx in zip(sdata.obs["groups"].cat.categories, group_indices_list):
    print(f"  group={g:>20s}  n_cells={len(idx)}")

## 5. Train spVIPES with label-based PoE over all groups

Passing a `label_key` selects the **label-based Product-of-Experts**
variant, which generalises to an arbitrary number of groups. Each group
gets its own encoder and private latent; the shared latent is obtained
by a Product of Experts across groups and is regularised to match cells
that share the same `cell.subtypes` label.

In [ ]:
N_SHARED   = 15
N_PRIVATE  = 5
N_HIDDEN   = 128
DROPOUT    = 0.1
MAX_EPOCHS = 100
BATCH_SIZE = 128
KL_WARMUP  = 30

torch.manual_seed(SEED)
np.random.seed(SEED)

model = spVIPES.model.spVIPES(
    sdata,
    n_hidden=N_HIDDEN,
    n_dimensions_shared=N_SHARED,
    n_dimensions_private=N_PRIVATE,
    dropout_rate=DROPOUT,
)
print(model)

In [ ]:
model.train(
    group_indices_list,
    max_epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    train_size=0.9,
    early_stopping=False,
    n_epochs_kl_warmup=KL_WARMUP,
)

In [ ]:
plt.plot(model.history["elbo_train"]["elbo_train"], label="train")
if "elbo_validation" in model.history:
    plt.plot(model.history["elbo_validation"]["elbo_validation"], label="val")
plt.xlabel("Epoch")
plt.ylabel("ELBO")
plt.title("spVIPES — multi-group training")
plt.legend()
plt.show()

## 6. Shared latent — cell-type identity across clinical groups

The **shared** space is designed to capture biology that every group
has in common. For a PBMC dataset stratified by `clinical.status`, that
should be **cell-type identity**: cell types separate cleanly, while
the different clinical groups **mix**.

In [ ]:
latents = model.get_latent_representation(group_indices_list, batch_size=BATCH_SIZE)
print("Latent keys:", list(latents.keys()))


def stitch_shared(sdata, latents):
    """Place per-group shared latents back into the combined obs order."""
    n_obs = sdata.n_obs
    dim = latents["shared_reordered"][0].shape[1]
    out = np.zeros((n_obs, dim), dtype=np.float32)
    for gi, g_positions in enumerate(sdata.uns["groups_obs_indices"]):
        out[g_positions] = latents["shared_reordered"][gi]
    return out


sdata.obsm["X_shared"] = stitch_shared(sdata, latents)

sc.pp.neighbors(sdata, use_rep="X_shared", n_neighbors=15)
sc.tl.umap(sdata, random_state=0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc.pl.umap(sdata, color=LABEL_KEY, ax=axes[0], show=False,
           title="Shared latent — cell type")
sc.pl.umap(sdata, color=GROUP_KEY, ax=axes[1], show=False,
           title="Shared latent — clinical.status")
plt.tight_layout()
plt.show()

If the model succeeds, the left panel should show well-separated
cell-type clusters and the right panel should show **each cluster
populated by cells from every clinical group** — i.e. identity is
preserved and the clinical covariate is integrated out.

## 6b. Disentanglement objective

spVIPES supports an optional **disentanglement objective** (inspired by
CellDISECT and Multi-ContrastiveVAE) that explicitly enforces:

* `z_shared` should encode cell-type biology but **not** clinical status
* `z_private` should encode clinical-status variation but **not** cell-type biology

It is realised through 4 auxiliary classifiers (2 adversarial, 2 supervised)
plus an optional InfoNCE on the shared latent. Enable the full configuration
via `disentangle_preset='full'`; see `disentangle_ablation.ipynb` for a
systematic component-by-component ablation.


In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

model_dis = spVIPES.model.spVIPES(
    sdata,
    n_hidden=N_HIDDEN,
    n_dimensions_shared=N_SHARED,
    n_dimensions_private=N_PRIVATE,
    dropout_rate=DROPOUT,
    disentangle_preset="full",
)
model_dis.train(
    group_indices_list,
    max_epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    train_size=0.9,
    early_stopping=False,
    n_epochs_kl_warmup=KL_WARMUP,
)


In [ ]:
latents_dis = model_dis.get_latent_representation(group_indices_list, batch_size=BATCH_SIZE)
sdata.obsm["X_shared_dis"] = stitch_shared(sdata, latents_dis)

sc.pp.neighbors(sdata, use_rep="X_shared_dis", n_neighbors=15, key_added="dis")
sc.tl.umap(sdata, random_state=0, neighbors_key="dis")
sdata.obsm["X_umap_shared_dis"] = sdata.obsm["X_umap"].copy()


### Compare baseline vs `disentangle_preset='full'`

Two diagnostics on the **shared** latent:

1. **UMAP** coloured by clinical group — disentanglement should mix groups
   more thoroughly while keeping cell-type structure intact.
2. **k-NN metrics** — group-mixing (higher = better) and label-purity
   (higher = better cell-type clustering).


In [ ]:
# Re-fit baseline UMAP from X_shared so the colour palette matches
sc.pp.neighbors(sdata, use_rep="X_shared", n_neighbors=15, key_added="base")
sc.tl.umap(sdata, random_state=0, neighbors_key="base")
sdata.obsm["X_umap_shared_base"] = sdata.obsm["X_umap"].copy()

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for col, (basis, title) in enumerate([
    ("X_umap_shared_base", "baseline"),
    ("X_umap_shared_dis",  "disentangle=full"),
]):
    sc.pl.embedding(sdata, basis=basis, color=GROUP_KEY,
                    ax=axes[0, col], show=False,
                    title=f"shared / {title} — group")
    sc.pl.embedding(sdata, basis=basis, color=LABEL_KEY,
                    ax=axes[1, col], show=False,
                    title=f"shared / {title} — cell type")
plt.tight_layout(); plt.show()


In [ ]:
from sklearn.neighbors import NearestNeighbors
import pandas as pd

def knn_mixing(z, group_labels, k=20):
    nn = NearestNeighbors(n_neighbors=k+1).fit(z)
    _, idx = nn.kneighbors(z); idx = idx[:, 1:]
    g = np.asarray(group_labels)
    expected = pd.Series(g).value_counts(normalize=True).reindex(np.unique(g)).values
    chi = []
    for i in range(len(g)):
        observed = pd.Series(g[idx[i]]).value_counts(normalize=True).reindex(np.unique(g), fill_value=0).values
        chi.append(((observed - expected) ** 2 / (expected + 1e-9)).sum())
    return float(np.exp(-np.mean(chi)))

def label_purity(z, labels, k=20):
    nn = NearestNeighbors(n_neighbors=k+1).fit(z)
    _, idx = nn.kneighbors(z); idx = idx[:, 1:]
    l = np.asarray(labels)
    return float(np.mean([(l[idx[i]] == l[i]).mean() for i in range(len(l))]))

groups_arr = sdata.obs[GROUP_KEY].values
labels_arr = sdata.obs[LABEL_KEY].values
pd.DataFrame({
    'group_mixing_shared (higher=better)': [
        knn_mixing(sdata.obsm['X_shared'], groups_arr),
        knn_mixing(sdata.obsm['X_shared_dis'], groups_arr),
    ],
    'label_purity_shared (higher=better)': [
        label_purity(sdata.obsm['X_shared'], labels_arr),
        label_purity(sdata.obsm['X_shared_dis'], labels_arr),
    ],
}, index=['baseline', 'disentangle=full'])


## 7. Private latents — one per clinical group

spVIPES learns **one private latent per group**. This is where
group-specific biology is expected to live: in a DIALOGUE-style
analysis, those are the *multicellular programs* that distinguish the
patient cohorts. Because each group has its own private subspace, we
visualise them separately.

In [ ]:
groups = list(sdata.obs["groups"].cat.categories)
n_groups = len(groups)

fig, axes = plt.subplots(1, n_groups, figsize=(6 * n_groups, 5), squeeze=False)

for gi, g in enumerate(groups):
    g_positions = sdata.uns["groups_obs_indices"][gi]
    g_obs = sdata.obs.iloc[g_positions].copy()

    priv = ad.AnnData(
        X=latents["private_reordered"][gi],
        obs=g_obs,
    )
    sc.pp.neighbors(priv, use_rep="X", n_neighbors=15)
    sc.tl.umap(priv, random_state=0)

    sc.pl.umap(priv, color=LABEL_KEY, ax=axes[0, gi], show=False,
               title=f"Private — {g}")

plt.tight_layout()
plt.show()

The private-latent UMAPs are computed **within each clinical group**
and coloured by cell type. Any residual cell-type structure here
represents variation *specific to that group* — e.g. how a particular
cell type behaves in that clinical condition. This is the spVIPES
analogue of DIALOGUE's multicellular programs: variation that is not
shared across cohorts and therefore ends up in the private space.

## 8. Summary

- We used the **same PBMC dataset** and the **same grouping variable
  (`clinical.status`)** as the pertpy DIALOGUE tutorial, but solved the
  problem with spVIPES' Product-of-Experts VAE.
- The **label-based PoE** variant (`setup_anndata(label_key=...)`)
  handles **N ≥ 2 groups** natively — a single call, no pairwise loop.
- The **shared** latent successfully captured cell-type identity while
  mixing clinical groups; the **private** latents isolated
  group-specific variation, the spVIPES analogue of DIALOGUE's
  multicellular programs.

### What to try next

- Swap `clinical.status` for any other sample-level covariate
  (treatment, batch, tissue) — the same code scales to any number of
  groups.
- Enable a Neural Spline Flow prior (`use_nf_prior=True,
  nf_type="NSF"`) to allow a multi-modal shared latent — useful when
  cell-type structure is far from Gaussian.
- Use `get_latent_representation` outputs downstream: cluster the
  shared latent for integrated annotation, or correlate private-latent
  directions with genes to recover group-specific programs.

### References

- Jerber, J. *et al.* (2021). *Population-scale single-cell RNA-seq
  profiling across dopaminergic neuron differentiation.* Nature Genetics.
- Jerby-Arnon, L., & Regev, A. (2022). *DIALOGUE maps multicellular
  programs in tissue from single-cell or spatial transcriptomics data.*
  Nature Biotechnology.
- Novella-Rausell, C., Peters, D. J. M., & Mahfouz, A. (2023).
  *Integrative learning of disentangled representations in multi-view
  single-cell data.* bioRxiv.
